In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
sentence = "What is the room rent limit?"

vector = embeddings.embed_query(sentence)

print(f"Type: {type(vector)}")
print(f"Dimensions: {len(vector)}")
print(f"First 5 values: {vector[:5]}")

Type: <class 'list'>
Dimensions: 384
First 5 values: [-0.019663864746689796, -0.06024441123008728, -0.03079020231962204, -0.03902539238333702, -0.009582976810634136]


In [5]:
sentences = [
    "What is the room rent limit?",
    "How much is the room rent per day?",  # similar meaning
    "What is the waiting period for cancer?"  # different meaning
]

vectors = embeddings.embed_documents(sentences)

print(f"Number of vectors: {len(vectors)}")
print(f"Each vector size: {len(vectors[0])}")

Number of vectors: 3
Each vector size: 384


In [6]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

q = embeddings.embed_query("What is the room rent limit?")
s1 = embeddings.embed_query("How much is the room rent per day?")
s2 = embeddings.embed_query("What is the waiting period for cancer?")

print(f"Similar sentences score:   {cosine_similarity(q, s1):.4f}")  # should be high ~0.9+
print(f"Different sentences score: {cosine_similarity(q, s2):.4f}")  # should be lower

Similar sentences score:   0.8590
Different sentences score: 0.5235


In [7]:
import sys
import os
sys.path.append("..")

from app.services.document_loader import load_and_chunk

pdf_path = os.path.abspath("../data/raw/StarHealthAssureInsurancePolicy-Policy-wording.pdf")
chunks = load_and_chunk(pdf_path)
print(f"Total chunks to store: {len(chunks)}")

Total chunks to store: 324


Don't run this cell again!!!

In [8]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../data/processed/chroma"
)

print("Done! Chunks stored in ChromaDB.")

Done! Chunks stored in ChromaDB.


In [9]:
chroma_path = os.path.abspath("../data/processed/chroma")
print(f"ChromaDB files:")
for f in os.listdir(chroma_path):
    print(f"  {f}")

ChromaDB files:
  cef7c175-2346-4108-9d05-16365fee4923
  chroma.sqlite3


In [10]:
print(f"Total chunks in vectorstore: {vectorstore._collection.count()}")

Total chunks in vectorstore: 324


In [11]:
# Simulate restarting — load from disk instead of recreating
vectorstore2 = Chroma(
    persist_directory="../data/processed/chroma",
    embedding_function=embeddings
)

print(f"Loaded from disk: {vectorstore2._collection.count()} chunks")

Loaded from disk: 324 chunks


In [12]:
query = "What is the waiting period?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source']}, Page: {doc.metadata['page']}")
    print(doc.page_content)


--- Result 1 ---
Source: c:\Users\srjsu\Downloads\policypal\data\raw\StarHealthAssureInsurancePolicy-Policy-wording.pdf, Page: 18
Waiting Period for Delivery - Normal 
Delivery and Caesarean Section
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst

--- Result 2 ---
Source: c:\Users\srjsu\Downloads\policypal\data\raw\StarHealthAssureInsurancePolicy-Policy-wording.pdf, Page: 13
"Waiting Periods", "Speciﬁc Waiting Periods" in the event of payment of premium 
within the stipulated grace Period.
iv. No interest will be charged lf the instalment premium is not paid on due date.
v. 
ln case of instalment premium due not recei

In [13]:
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n--- Result {i+1} | Score: {score:.4f} ---")
    print(f"Page: {doc.metadata['page']}")
    print(doc.page_content)


--- Result 1 | Score: 0.5668 ---
Page: 18
Waiting Period for Delivery - Normal 
Delivery and Caesarean Section
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst

--- Result 2 | Score: 0.5743 ---
Page: 13
"Waiting Periods", "Speciﬁc Waiting Periods" in the event of payment of premium 
within the stipulated grace Period.
iv. No interest will be charged lf the instalment premium is not paid on due date.
v. 
ln case of instalment premium due not received within the grace period, the 
policy will get cancelled.
vi. ln the event of a claim, all subsequent premium instalments shall immediately 
become due and payable.
vii. The

In [14]:
test_queries = [
    "What is the waiting period for pre-existing diseases?",
    "What is the room rent limit per day?",
    "Is maternity covered?",
    "What are the exclusions?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    results = vectorstore.similarity_search_with_score(query, k=2)
    for doc, score in results:
        print(f"\nScore: {score:.4f} | Page: {doc.metadata['page']}")
        print(doc.page_content[:200])


Query: What is the waiting period for pre-existing diseases?

Score: 0.4466 | Page: 4
advice of a Medical Practitioner. The Insured is not admitted as a day care or in-patient.
Pre-Existing Disease: Pre-existing Disease means any condition, ailment, injury or 
disease:
a) 
That is/are 

Score: 0.4530 | Page: 11
waiting period of 24 months from the date of inception of this policy
Note: The above mentioned waiting period will not apply for treatment related to 
congenital Internal disease / defects for the Un

Query: What is the room rent limit per day?

Score: 0.4974 | Page: 5
5,000
10,000
25,00,000
5,000
10,000
50,00,000
5,000
10,000
75,00,000
8,000
15,000
1,00,00,000
8,000
15,000
2,00,00,000
8,000
15,000
Sum Insured
in lakhs (Rs.)
5
10/15/20/25
50/75/100/200
Room Rent 
Cr

Score: 0.6175 | Page: 18
19 of 20
Unique Identiﬁcation No.: SHAHLIP23131V022223
Star Health Assure Insurance Policy
POL / SHA / V.2 / 2022
SCHEDULE OF BENEFITS
S.No
Sum Insured (INR)
5 lacs
10 lacs
15 lacs
20 lacs